## PROYECTO FINAL
## Curso: Inteligencia Artificial Generativa
## Alumnos:
## Christian Gutiérrez
## Gianella Fortini

## 1. INSTALAR LIBRERIAS

In [1]:
# Instala las siguientes librerias:

# Transformes: Biblioteca de Hugging Face para trabajar con modelos de IA. Sirve para: i)Descargar modelos LLM, ii)Ejecutar inferencia, iii) Hacer fine-tuning
# Accelerate: Librería para optimizar el uso de GPU/CPU. Sirve para: i)Cargar modelos grandes, ii) Manejar memoria, iii) Distribuir inferencia
# Bitsandbytes: Permite cuantizar modelos para que usen menos memoria. Sirve para: i) Modelo de 16 bits → 8 bits, ii) Modelo de 8 bits → 4 bits
# Sentencepiece: Tokenizer usado por muchos LLM.

# faiss-cpu: FAISS = Facebook AI Similarity Search. Sirve para: i) búsqueda vectorial, ii) nearest neighbors, iii) retrieval en RAG
# sentence-transformes: Sirve para crear embeddings de texto.
# datasets: Librería de Hugging Face para manejar datasets. Sirve para: i) descargar datasets, ii) cargar CSV, iii) dividir train/test, iv) streaming de datos

!pip install -U transformers accelerate bitsandbytes sentencepiece ir_datasets
!pip install faiss-cpu sentence-transformers datasets
!pip install -q transformers accelerate sentence-transformers evaluate rouge_score rank_bm25

## 2. Descargar datasets

MS Marco , creado por Microsoft, se utiliza en proyectos de búsqueda de información y RAG (Retrieval-Augmented Generation) porque proporciona un conjunto de datos realista de consultas y documentos que permite evaluar y entrenar sistemas que recuperan información relevante.
Ventajas de su uso:
i.	Permite trabajar con datos reales de búsqueda.
ii.	Es el benchmark más usado en investigación de búsqueda semántica.
iii.	Permite probar piplines completos de RAG.
Es ideal para practicar chunking y ovelap debido a que los documentos puden ser largos.

In [2]:
from datasets import load_dataset

ds = load_dataset("microsoft/ms_marco", "v1.1")

documents = []
metadata = []

sample = ds["train"].select(range(2000))

for row in sample:
    query_id = row["query_id"]
    query = row["query"]

    texts = row["passages"]["passage_text"]
    selected_flags = row["passages"].get("is_selected", [0] * len(texts))

    for j, text in enumerate(texts):
        if text and text.strip() and selected_flags[j] == 1:
            documents.append(text)
            metadata.append({
                "query_id": query_id,
                "query": query,
                "passage_idx": j,
                "is_selected": selected_flags[j]
            })

print("Cantidad de textos extraídos:", len(documents))
print(documents[0][:1000])
print(metadata[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Cantidad de textos extraídos: 2157
Results-Based Accountability® (also known as RBA) is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole. RBA is also used by organizations to improve the performance of their programs. Creating Community Impact with RBA. Community impact focuses on conditions of well-being for children, families and the community as a whole that a group of leaders is working collectively to improve. For example: “Residents with good jobs,” “Children ready for school,” or “A safe and clean neighborhood”.
{'query_id': 19699, 'query': 'what is rba', 'passage_idx': 5, 'is_selected': 1}


## 3. Chunking + Overlap

El chunking y el overlap son técnicas cuando se construye un sistema RAG o cualquier sistema de búsqueda semántica con embeddings. Su objetivo es mejorar la recuperación de información cuando los documentos son largos.

Objetivo del chunking

Chunking significa dividir documentos grandes en fragmentos pequeños llamados chunks, y se ejecutan debido a las limitaciones en la ejecución de embeddings y los LLMs (Large Language Models):

i. Límite de tokens: No se pueden procesar documentos muy largos.

ii. Mayor precisión de retrieval: Buscar dentro de fragmentos pequeños mejora la probabilidad de encontrar información relevante.

iii. Embeddings más representativos: Un embedding funciona mejor cuando el texto es temáticamente coherente.

Objetivo del overlap

Overlap significa que los chunks comparten una parte del texto entre sí. El problema del chunking simple es que puede cortar información importante en la mitad de una idea.

In [3]:
def chunk_text(doc, chunk_size=80, overlap=20):
    words = doc.split()
    chunks = []
    step = chunk_size - overlap

    if step <= 0:
        raise ValueError("chunk_size must be greater than overlap")

    for start in range(0, len(words), step):
        chunk = " ".join(words[start:start + chunk_size]).strip()
        if chunk:
            chunks.append(chunk)

    return chunks

# Ejecución del chunking
chunked_documents = []
chunked_metadata = []

for doc, meta in zip(documents, metadata):
    chunks = chunk_text(doc, chunk_size=80, overlap=20)
    for k, chunk in enumerate(chunks):
        chunked_documents.append(chunk)
        chunked_metadata.append({**meta, "chunk_id": j})

print("Chunks generados:", len(chunked_documents))
print(chunked_documents[0])
print(chunked_metadata[0])

Chunks generados: 3579
Results-Based Accountability® (also known as RBA) is a disciplined way of thinking and taking action that communities can use to improve the lives of children, youth, families, adults and the community as a whole. RBA is also used by organizations to improve the performance of their programs. Creating Community Impact with RBA. Community impact focuses on conditions of well-being for children, families and the community as a whole that a group of leaders is working collectively to improve. For example:
{'query_id': 19699, 'query': 'what is rba', 'passage_idx': 5, 'is_selected': 1, 'chunk_id': 7}


## 4. Embeddings

Los embeddings son una representación matemática del lenguaje que permite que los modelos de IA comprendan el significado semántico del texto. Un embedding es un vector numérico que representa el significado de una palabra, frase o documento en un espacio multidimensional.

El objetivo principal es convertir texto en números que capturen significado para que los algoritmos puedan compararlos. Los objetivos son los siguientes:

i. Medir similitud semántica

ii. Permitir búsqueda semántica

iii. Reducir alucinaciones

Se utiliza el modelo "BAAI/bge-large-en-v1.5" debido a las siguientes ventajas:

✔ muy alto rendimiento en retrieval

✔ entrenado específicamente para dense retrieval

✔ funciona muy bien con FAISS y rerankers

In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-large-en-v1.5")
embeddings = embedder.encode(chunked_documents, normalize_embeddings=True)
embeddings = np.asarray(embeddings).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64
index.add(embeddings)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 5. FAISS HNSW

In [5]:
import faiss
import numpy as np

# embeddings: matriz numpy de tamaño (n, d)
embeddings = np.asarray(embeddings).astype("float32")

dim = embeddings.shape[1]

# Normalizar si usarás cosine similarity
faiss.normalize_L2(embeddings)

# Crear índice HNSW
M = 32
index = faiss.IndexHNSWFlat(dim, M, faiss.METRIC_INNER_PRODUCT)

# Ajustar parámetros
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64

# Agregar embeddings
index.add(embeddings)

print("Total vectores indexados:", index.ntotal)

Total vectores indexados: 3579


## 6. Retrieval

In [6]:
def retrieve(query, k=8):
    query_for_embedding = f"Represent this sentence for searching relevant passages: {query}"
    q_emb = embedder.encode(
        [query_for_embedding],
        normalize_embeddings=True
    ).astype("float32")

    k = min(k, index.ntotal)
    D, I = index.search(q_emb, k)

    print("Raw indices from FAISS:", I[0])
    print("Raw scores from FAISS:", D[0])

    results = []
    scores = []

    for idx, score in zip(I[0], D[0]):
        if idx != -1 and 0 <= idx < len(chunked_documents):
            results.append(chunked_documents[idx])
            scores.append(float(score))

    print("Retrieved valid docs:", len(results))
    return results, scores


## 7. Reranker

In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, retrieved_chunks, top_n=3):
    if not retrieved_chunks:
        return []

    valid_chunks = [c for c in retrieved_chunks if isinstance(c, str) and c.strip()]
    if not valid_chunks:
        return []

    pairs = [[query, chunk] for chunk in valid_chunks]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(valid_chunks, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 8. LLM Qwen

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

def build_context(reranked_docs):
    parts = []
    for i, item in enumerate(reranked_docs, start=1):
        doc = item[0] if isinstance(item, tuple) else item
        parts.append(f"Source {i}: {doc}")
    return "\n\n".join(parts)

def generate_answer(query, reranked_docs):
    if not reranked_docs:
        return "No relevant context was retrieved."

    context = build_context(reranked_docs)

    prompt = f"""Context:
{context}

Question: {query}

Answer in 2 or 3 short sentences using only the information in the context.
Avoid extra interpretation or broad claims.
Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.0,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [9]:
for i in range(20):
    print(i, ds["train"][i]["query"])

0 what is rba
1 was ronald reagan a democrat
2 how long do you need for sydney and surrounding areas
3 price to install tile in shower
4 why conversion observed in body
5 where are the lungs located in the back
6 cost to get a patent
7 what does a metabolic acidosis need to reverse the condition
8 best tragedies of ancient greece
9 what is a conifer
10 in animals somatic cells are produced by and gametic cells are produced by
11 remembering the name of the author who wrote the cat in the hat
12 how long cooking chicken legs in the big easy
13 average cost of heating per square foot
14 is mount pinatubo made of granite or basalt
15 concrete pads cost
16 what kind of organism is a black damsel
17 who coined the phrase it is what it is
18 what is oilskin fabric
19 how long is german measles contagious


In [10]:
query = "Who is Sophocles?"

retrieved_docs, scores = retrieve(query, k=8)

reranked_docs = rerank(query, retrieved_docs, top_n=3)

answer = generate_answer(query, reranked_docs)

print(answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw indices from FAISS: [  12 2674  932 3428 3056  477 3084  298]
Raw scores from FAISS: [0.52806664 0.513274   0.49493027 0.4692788  0.4673364  0.46599138
 0.46066076 0.45379946]
Retrieved valid docs: 8
Sophocles was a famous playwright of ancient Greek tragedy, alongside Aeschylus and Euripides. His works are still performed centuries after his initial premiere.


## 9. Citation per sentence

In [11]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(answer)

import re
import numpy as np

def citation_per_sentence(answer, reranked_docs):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer) if s.strip()]
    chunks = [doc for doc, score in reranked_docs if isinstance(doc, str) and doc.strip()]

    if not sentences:
        return answer

    if not chunks:
        return "\n".join([f"{s} [source unavailable]" for s in sentences])

    sent_emb = embedder.encode(sentences, normalize_embeddings=True)
    chunk_emb = embedder.encode(chunks, normalize_embeddings=True)

    # Asegurar 2D
    sent_emb = np.atleast_2d(sent_emb)
    chunk_emb = np.atleast_2d(chunk_emb)

    sim = sent_emb @ chunk_emb.T
    citations = sim.argmax(axis=1)

    results = []
    for i, s in enumerate(sentences):
        source = int(citations[i]) + 1
        results.append(f"{s} [source {source}]")

    return "\n".join(results)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
query = "Who is Sophocles?"

final_answer = citation_per_sentence(answer, reranked_docs)
print(final_answer)

Sophocles was a famous playwright of ancient Greek tragedy, alongside Aeschylus and Euripides. [source 1]
His works are still performed centuries after his initial premiere. [source 1]


## 10. Hallucination guard

In [13]:
import numpy as np
import re

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def semantic_hallucination_guard(answer, context, threshold=0.40):

    answer_sentences = split_sentences(answer)
    context_sentences = split_sentences(context)

    if not answer_sentences or not context_sentences:
        return {
            "mean_similarity": 0.0,
            "hallucination_ratio": 0.0,
            "details": []
        }

    answer_emb = embedder.encode(answer_sentences, normalize_embeddings=True)
    context_emb = embedder.encode(context_sentences, normalize_embeddings=True)

    similarity_matrix = answer_emb @ context_emb.T

    best_scores = similarity_matrix.max(axis=1)
    best_idx = similarity_matrix.argmax(axis=1)

    hallucinated = 0
    details = []

    for sent, score, idx in zip(answer_sentences, best_scores, best_idx):

        is_hallucinated = float(score) < threshold

        if is_hallucinated:
            hallucinated += 1

        details.append({
            "sentence": sent,
            "similarity": float(score),
            "hallucinated": is_hallucinated,
            "best_context_sentence": context_sentences[int(idx)]
        })

    return {
        "mean_similarity": float(np.mean(best_scores)),
        "hallucination_ratio": hallucinated / len(answer_sentences),
        "details": details
    }

In [14]:
context = build_context(reranked_docs)

result = semantic_hallucination_guard(answer, context)

print("Mean similarity:", result["mean_similarity"])
print("Hallucination ratio:", result["hallucination_ratio"])

for item in result["details"]:
    print("\nSentence:", item["sentence"])
    print("Similarity:", item["similarity"])
    print("Hallucinated:", item["hallucinated"])
    print("Best context sentence:", item["best_context_sentence"])

Mean similarity: 0.7427419424057007
Hallucination ratio: 0.0

Sentence: Sophocles was a famous playwright of ancient Greek tragedy, alongside Aeschylus and Euripides.
Similarity: 0.7850791215896606
Hallucinated: False
Best context sentence: The most famous playwrights of the genre were Aeschylus, Sophocles, and Euripides and many of their works were still performed centuries after their initial premiere.

Sentence: His works are still performed centuries after his initial premiere.
Similarity: 0.700404703617096
Hallucinated: False
Best context sentence: The most famous playwrights of the genre were Aeschylus, Sophocles, and Euripides and many of their works were still performed centuries after their initial premiere.


## 11. Grounding evaluation

In [15]:
def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

def grounding_evaluation(answer, reranked_docs, embedder, threshold=0.40):
    cleaned_answer = re.sub(r'\[Source\s+\d+\]', '', answer)
    answer_sentences = split_sentences(cleaned_answer)

    source_sentences = []
    source_map = []

    for source_id, (doc, score) in enumerate(reranked_docs, start=1):
        for sent in split_sentences(doc):
            if len(sent.split()) >= 5:
                source_sentences.append(sent)
                source_map.append(source_id)

    if not answer_sentences or not source_sentences:
        return {
            "mean_score": 0.0,
            "grounded_ratio": 0.0,
            "details": []
        }

    ans_emb = embedder.encode(answer_sentences, normalize_embeddings=True)
    src_emb = embedder.encode(source_sentences, normalize_embeddings=True)

    sim_matrix = ans_emb @ src_emb.T
    best_scores = sim_matrix.max(axis=1)
    best_idx = sim_matrix.argmax(axis=1)

    details = []
    grounded_count = 0

    for sent, score, idx in zip(answer_sentences, best_scores, best_idx):
        grounded = float(score) >= threshold
        if grounded:
            grounded_count += 1

        details.append({
            "sentence": sent,
            "score": float(score),
            "grounded": grounded,
            "source_id": source_map[int(idx)],
            "source_text": source_sentences[int(idx)]
        })

    return {
        "mean_score": float(np.mean(best_scores)),
        "grounded_ratio": grounded_count / len(answer_sentences),
        "details": details
    }

In [16]:
evaluation = grounding_evaluation(answer, reranked_docs, embedder, threshold=0.50)

print("Mean grounding score:", evaluation["mean_score"])
print("Grounded ratio:", evaluation["grounded_ratio"])

for item in evaluation["details"]:
    print(item["sentence"])
    print("Score:", item["score"])
    print("Grounded:", item["grounded"])
    print("Source:", item["source_id"])
    print("Best source sentence:", item["source_text"])
    print("------")

Mean grounding score: 0.7427419424057007
Grounded ratio: 1.0
Sophocles was a famous playwright of ancient Greek tragedy, alongside Aeschylus and Euripides.
Score: 0.7850791215896606
Grounded: True
Source: 1
Best source sentence: The most famous playwrights of the genre were Aeschylus, Sophocles, and Euripides and many of their works were still performed centuries after their initial premiere.
------
His works are still performed centuries after his initial premiere.
Score: 0.700404703617096
Grounded: True
Source: 1
Best source sentence: The most famous playwrights of the genre were Aeschylus, Sophocles, and Euripides and many of their works were still performed centuries after their initial premiere.
------


## 12. BLEU/ROUGE

In [17]:
import evaluate

bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

def evaluate_generation(prediction, reference):
    rouge_results = rouge_metric.compute(
        predictions=[prediction],
        references=[reference]
    )

    bleu_results = bleu_metric.compute(
        predictions=[prediction],
        references=[[reference]]
    )

    return {
        "bleu": bleu_results["bleu"],
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "rougeLsum": rouge_results["rougeLsum"]
    }

In [18]:
predictions = [
    "Sophocles was a famous playwright of ancient Greek tragedy, alongside Aeschylus and Euripides.",
    "His works continued to be performed centuries after their original premiere."
]

references = [
    "Sophocles was one of the most famous playwrights of ancient Greek tragedy, along with Aeschylus and Euripides.",
    "Many of his works were still performed centuries after their initial premiere."
]

rouge_results = rouge_metric.compute(
    predictions=predictions,
    references=references
)

bleu_results = bleu_metric.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("BLEU:", bleu_results["bleu"])
print("ROUGE:", rouge_results)

BLEU: 0.3108390619195697
ROUGE: {'rouge1': np.float64(0.6376811594202899), 'rouge2': np.float64(0.40476190476190477), 'rougeL': np.float64(0.6376811594202899), 'rougeLsum': np.float64(0.6376811594202899)}


## 13. Hybrid BM25 + FAISS

In [25]:
import numpy as np
import re
from rank_bm25 import BM25Okapi

# --------------------------------------------------
# 1. Limpieza/tokenización más robusta
# --------------------------------------------------

stopwords = {
    "who", "what", "was", "is", "the", "a", "an", "of", "in", "on", "and",
    "to", "for", "with", "by", "at", "from", "as", "did", "does", "do"
}

def tokenize_text(text):
    tokens = re.findall(r"\b\w+\b", text.lower())
    return [t for t in tokens if t not in stopwords]

# Corpus BM25
tokenized_corpus = [tokenize_text(doc) for doc in chunked_documents]
bm25 = BM25Okapi(tokenized_corpus)

# --------------------------------------------------
# 2. Normalización segura
# --------------------------------------------------

def minmax_normalize(scores):
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0:
        return scores
    smin, smax = scores.min(), scores.max()
    if smax == smin:
        return np.ones_like(scores)
    return (scores - smin) / (smax - smin)

# --------------------------------------------------
# 3. Hybrid retrieval CORREGIDO
# --------------------------------------------------

def hybrid_retrieve(query, k_bm25=10, k_faiss=10, top_n=10, alpha=0.35):
    # -------- BM25 --------
    bm25_tokens = tokenize_text(query)
    bm25_scores = np.asarray(bm25.get_scores(bm25_tokens), dtype=float)

    bm25_idx = np.argsort(bm25_scores)[::-1][:k_bm25]

    # -------- FAISS --------
    query_for_embedding = f"Represent this sentence for searching relevant passages: {query}"
    query_vec = embedder.encode(
        [query_for_embedding],
        normalize_embeddings=True
    ).astype("float32")

    D, I = index.search(query_vec, min(k_faiss, index.ntotal))
    faiss_idx = I[0]
    faiss_scores = D[0]

    # Crear vector global FAISS del tamaño del corpus
    faiss_full = np.zeros(len(chunked_documents), dtype=float)
    for idx, score in zip(faiss_idx, faiss_scores):
        if idx != -1 and 0 <= idx < len(chunked_documents):
            faiss_full[idx] = float(score)

    # -------- Unir candidatos --------
    candidates = sorted(set(bm25_idx).union(set(int(i) for i in faiss_idx if i != -1)))

    if not candidates:
        return []

    # Tomar scores solo de candidatos
    bm25_candidate_scores = np.array([bm25_scores[idx] for idx in candidates], dtype=float)
    faiss_candidate_scores = np.array([faiss_full[idx] for idx in candidates], dtype=float)

    # Normalizar dentro del conjunto candidato
    bm25_norm = minmax_normalize(bm25_candidate_scores)
    faiss_norm = minmax_normalize(faiss_candidate_scores)

    # Combinar
    results = []
    for pos, idx in enumerate(candidates):
        combined_score = alpha * bm25_norm[pos] + (1 - alpha) * faiss_norm[pos]

        results.append({
            "idx": idx,
            "bm25_score": float(bm25_scores[idx]),
            "faiss_score": float(faiss_full[idx]),
            "bm25_norm": float(bm25_norm[pos]),
            "faiss_norm": float(faiss_norm[pos]),
            "combined_score": float(combined_score),
            "document": chunked_documents[idx]
        })

    results = sorted(results, key=lambda x: x["combined_score"], reverse=True)
    return results[:top_n]

# --------------------------------------------------
# 4. Reranker CONSISTENTE
#    Devuelve [(doc, score), ...]
# --------------------------------------------------

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=3):
    valid_docs = [doc for doc in docs if isinstance(doc, str) and doc.strip()]
    if not valid_docs:
        return []

    pairs = [[query, doc] for doc in valid_docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(valid_docs, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# --------------------------------------------------
# 5. Ejecución del pipeline híbrido + reranking
# --------------------------------------------------

query = "Who was Sophocles?"

hybrid_results = hybrid_retrieve(
    query=query,
    k_bm25=10,
    k_faiss=10,
    top_n=10,
    alpha=0.35
)

print("Query:", query)
print("\nTop results using corrected Hybrid BM25 + FAISS retrieval")

for rank, item in enumerate(hybrid_results[:5], start=1):
    print("\n" + "=" * 70)
    print(f"Rank: {rank}")
    print("Index:", item["idx"])
    print("BM25 raw:", round(item["bm25_score"], 4))
    print("FAISS raw:", round(item["faiss_score"], 4))
    print("BM25 norm:", round(item["bm25_norm"], 4))
    print("FAISS norm:", round(item["faiss_norm"], 4))
    print("Combined:", round(item["combined_score"], 4))
    print("Document preview:")
    print(item["document"][:350])

retrieved_docs = [item["document"] for item in hybrid_results]
reranked_docs = rerank(query, retrieved_docs, top_k=3)

print("\n\nTop reranked results")
for i, (doc, score) in enumerate(reranked_docs, start=1):
    print(f"\n--- Reranked result {i} | score={score:.4f} ---")
    print(doc[:400])

answer = generate_answer(query, reranked_docs)

print("\nGenerated answer:\n")
print(answer)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Who was Sophocles?

Top results using corrected Hybrid BM25 + FAISS retrieval

Rank: 1
Index: 12
BM25 raw: 6.2184
FAISS raw: 0.5514
BM25 norm: 1.0
FAISS norm: 1.0
Combined: 1.0
Document preview:
by Mark Cartwright. Greek tragedy was a popular and influential form of drama performed in theatres across ancient Greece from the late 6th century BCE. The most famous playwrights of the genre were Aeschylus, Sophocles, and Euripides and many of their works were still performed centuries after their initial premiere. Greek tragedy led to Greek com

Rank: 2
Index: 2674
BM25 raw: 0.0
FAISS raw: 0.5199
BM25 norm: 0.0
FAISS norm: 0.9429
Combined: 0.6129
Document preview:
a dialogue. Vyasa, the narrator of the Mahabharata, is traditionally considered the compiler of the Puranas.

Rank: 3
Index: 932
BM25 raw: 0.0
FAISS raw: 0.5181
BM25 norm: 0.0
FAISS norm: 0.9395
Combined: 0.6107
Document preview:
Odysseus talked to his mother Anticlea, who died of grief when he did not return home after the